In [8]:
!pip install opencv-python


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import cv2
import numpy as np
import os

IMAGE_PATH = os.path.join("images", "cake.jpg")
img = cv2.imread(IMAGE_PATH)
if img is None:
    raise FileNotFoundError(f"Could not load image at: {IMAGE_PATH}")

h, w = img.shape[:2]
img = img[0:int(0.72*h), int(0.05*w):int(0.95*w)]

cv2.imwrite("01_input.jpg", img)
print("Saved: 01_input.jpg", img.shape)

Saved: 01_input.jpg (864, 1440, 3)


In [10]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)

cv2.imwrite("02_gray.jpg", gray)
print("Saved: 02_gray.jpg")

Saved: 02_gray.jpg


In [11]:
CANNY_LOW = 30
CANNY_HIGH = 100

edges = cv2.Canny(blur, CANNY_LOW, CANNY_HIGH)
cv2.imwrite("03_edges.jpg", edges)

print("Saved: 03_edges.jpg | Non-zero edges:", np.count_nonzero(edges))

Saved: 03_edges.jpg | Non-zero edges: 50750


In [12]:
kernel = np.ones((3, 3), np.uint8)
edges_closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=1)

cv2.imwrite("04_edges_closed.jpg", edges_closed)
print("Saved: 04_edges_closed.jpg | Non-zero edges:", np.count_nonzero(edges_closed))

Saved: 04_edges_closed.jpg | Non-zero edges: 67253


In [ ]:
MIN_CONTOUR_AREA = 800
TOP_K = 8 

contours, _ = cv2.findContours(edges_closed.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours = sorted(contours, key=cv2.contourArea, reverse=True)

overlay = img.copy()
kept = 0

for c in contours[:TOP_K]:
    if cv2.contourArea(c) >= MIN_CONTOUR_AREA:
        kept += 1
        cv2.drawContours(overlay, [c], -1, (0, 255, 0), 2)

cv2.imwrite("05_final_contours_overlay.jpg", overlay)
print("Contours drawn:", kept)
print("Saved: 05_final_contours_overlay.jpg")

Contours drawn: 8
Saved: 05_final_contours_overlay.jpg
